# 16 — Azure API Management as an AI Gateway for MCP

Notebooks 01-15 ran the Azure MCP Server as a *local* process on your
machine, launched over stdio by `npx @azure/mcp@latest`. This second part
of the course covers a different, complementary pattern: running MCP
servers (and LLM backends) *behind* [Azure API Management (APIM)](https://learn.microsoft.com/en-us/azure/api-management/api-management-key-concepts),
using its **AI gateway** capabilities to centrally secure, govern, and
observe them.

This is not a replacement for the Azure MCP Server -- it's how you'd expose
*your own* APIs and tools (or a fleet of them) as governed MCP servers for
GitHub Copilot, Claude, ChatGPT, or any other MCP client, instead of
handing out raw backend URLs and API keys.

What you'll learn in notebooks 16-22:

- What the AI gateway in API Management is and why it exists.
- How to expose a managed REST API as a remote MCP server (17).
- How to secure inbound and outbound access with Microsoft Entra ID and
  API Management's credential manager (18).
- How to govern MCP server tools with policies: rate limits, caching,
  tracing (19).
- How to federate existing MCP servers and discover them via Azure API
  Center (20).
- How the same gateway fronts LLM backends: load balancing, token limits,
  semantic caching (21).
- A capstone agent that calls both an LLM and MCP tools entirely through
  the gateway (22).

Read alongside [docs/10_apim_ai_gateway_and_mcp.md](../docs/10_apim_ai_gateway_and_mcp.md).

## MCP architecture, recapped

[Azure API Management's MCP documentation](https://learn.microsoft.com/en-us/azure/api-management/mcp-server-overview)
describes the same architecture notebook 01 introduced, using API
Management's own terms:

| Component | Description |
| --- | --- |
| **MCP hosts** | LLM applications (chat apps, IDE assistants) that need external capabilities |
| **MCP clients** | Protocol clients inside the host, one per server connection |
| **MCP servers** | Programs that expose tools/prompts/resources to clients |
| **MCP protocol** | JSON-RPC 2.0 over a defined transport |

Remote MCP servers (the kind API Management exposes) use HTTP-based
transports:

| Transport | Endpoint(s) | Notes |
| --- | --- | --- |
| Streamable HTTP | `/mcp` | Current transport; what API Management uses |
| SSE (deprecated) | `/sse` + `/messages` | Legacy HTTP+SSE transport from protocol `2024-11-05` |

So an API Management-hosted MCP server URL typically looks like
`https://<apim-name>.azure-api.net/<mcp-server-path>/mcp`.

In [ ]:
# A quick illustration of how an APIM MCP server URL is put together -- no
# network call yet, just string plumbing you'll reuse for the rest of this
# notebook series.
apim_gateway_hostname = "my-apim-instance.azure-api.net"
mcp_server_path = "weather-mcp"  # the "Name" you gave the MCP server in the portal

mcp_server_url = f"https://{apim_gateway_hostname}/{mcp_server_path}/mcp"
print("MCP server (Streamable HTTP) endpoint:", mcp_server_url)

## Two ways API Management exposes MCP servers

| Source | Description |
| --- | --- |
| **REST API as MCP server** | Expose any REST API already managed in API Management (including ones imported from Azure resources) as an MCP server; selected operations become tools. Covered in notebook 17. |
| **Existing MCP server** | Proxy an MCP-compatible server hosted elsewhere (LangServe, Azure Functions, Azure Logic Apps, ...) through API Management. Covered in notebook 20. |

Under the hood, both are represented in Azure Resource Manager as an API
Management **API** resource with `type: mcp` -- the same resource kind used
for REST/SOAP/GraphQL APIs, just a different `type`. That means MCP servers
get all of API Management's existing machinery for free: policies,
products/subscriptions, diagnostics, and CI/CD via ARM/Bicep/Terraform/CLI
(notebook 20).

## AI gateway capability summary

The **AI gateway** is not a separate product -- it's a set of capabilities
layered on API Management's existing API gateway
([genai-gateway-capabilities](https://learn.microsoft.com/en-us/azure/api-management/genai-gateway-capabilities)):

| Capability | Examples |
| --- | --- |
| Traffic mediation | Expose REST APIs as MCP servers; passthrough to existing MCP servers; import A2A agent APIs |
| Security & safety | Managed identities to AI services; OAuth via credential manager; Entra ID token validation |
| Resiliency | Backend load balancing (round-robin/weighted/priority/session-aware); circuit breakers with dynamic trip duration |
| Observability & governance | Prompt/completion logging to Azure Monitor; per-consumer token metrics in Application Insights; token quotas |

Availability varies by **service tier**. MCP server support requires
Developer, Basic, Basic v2, Standard, Standard v2, or Premium/Premium v2 --
check the "Applies to" banner on each Microsoft Learn article before you
build against a specific tier.